# 6. Аналитика буста

**Анализируем на полном пуле кандидатов, не внутри топ-5.**

**Нужны:** все файлы из тетрадок 1–3

**Создаёт:** `boost_analytics.csv`

In [1]:
import csv, math, os, random
from collections import defaultdict, Counter

random.seed(42)
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def parse_vector(s):
    if not s or not s.strip(): return None
    return [float(x) for x in s.split(",")]

def parse_set(s):
    if not s or not s.strip(): return None
    return set(s.split("|"))

def parse_bool(s):
    return str(s).strip().lower() in ("true","1","yes")

DOMAINS = [r["domain_name"].strip()
           for r in load_csv(os.path.join(BASE_DIR,"ontology_domains.csv"))]

SIM_MATRIX = {}
with open(os.path.join(BASE_DIR,"ontology_domain_similarity.csv"),
          newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        d1 = row["domain"].strip()
        for d2 in DOMAINS:
            SIM_MATRIX[(d1,d2)] = float(row.get(d2,0))

ratings_by_id = {r["mentor_id"]: float(r["rating"])
                 for r in load_csv(os.path.join(BASE_DIR,"mentor_ratings.csv"))}

print(f"Областей: {len(DOMAINS)}  |  Рейтингов: {len(ratings_by_id)}")


Областей: 10  |  Рейтингов: 2000


In [2]:
def parse_mentee(r):
    return {
        "id":            r["id"],
        "name":          r["name"],
        "level_score":   int(r["level_score"]),
        "skills_missing": parse_bool(r["skills_missing"]),
        "language_set":  parse_set(r["language_set"]),
        "format_set":    parse_set(r["format_set"]),
        "domain_vector": parse_vector(r["domain_vector"]),
        "skills_list":   [s.strip() for s in
                          r.get("skills_normalized","").split(";") if s.strip()],
    }

def parse_mentor(r):
    return {
        "id":               r["id"],
        "name":             r["name"],
        "profession":       r["profession"],
        "domain":           r.get("domain",""),
        "level_score":      int(r["level_score"]),
        "level_raw":        r.get("level_raw",""),
        "language_set":     parse_set(r["language_set"]),
        "format_set":       parse_set(r["format_set"]),
        "domain_vector":    parse_vector(r["domain_vector"]),
        "skills_list":      [s.strip() for s in
                             r.get("skills_normalized","").split(";") if s.strip()],
        "experience_norm":  float(r["experience_norm"]),
        "experience_years": int(r.get("experience_years",0) or 0),
        "available":        parse_bool(r["available"]),
        "boosted":          parse_bool(r["boosted"]),
        "boost_k":          float(r["boost_k"]),
        "rating":           ratings_by_id.get(r["id"], 3.8),
    }

mentees = [parse_mentee(r)
           for r in load_csv(os.path.join(BASE_DIR,"mentees_processed.csv"))]
mentors = [parse_mentor(r)
           for r in load_csv(os.path.join(BASE_DIR,"mentors_processed.csv"))]

print(f"Mentees: {len(mentees)}  |  Mentors: {len(mentors)}")
print(f"Доступных менторов: {sum(1 for m in mentors if m['available'])}")


Mentees: 5000  |  Mentors: 2000
Доступных менторов: 1613


In [3]:
# ── 4 фактора: skill, domain, exp, rating (goal убран — вклад <0.5%) ─────────

def jaccard(s1, s2):
    a, b = set(s1), set(s2)
    if not a or not b: return None
    return len(a & b) / len(a | b)

def domain_sim_raw(mentee, mentor):
    v1, v2 = mentee["domain_vector"], mentor["domain_vector"]
    if not v1 or not v2: return 0.0
    return sum(
        v1[i] * SIM_MATRIX.get((d1,d2), 0) * v2[j]
        for i,d1 in enumerate(DOMAINS)
        for j,d2 in enumerate(DOMAINS)
    )

def sets_compat(s1, s2):
    if s1 is None or s2 is None: return True
    return len(s1 & s2) > 0

def hard_filter(mentee, pool):
    return [m for m in pool
            if m["available"]
            and m["level_score"] > mentee["level_score"]
            and sets_compat(mentee["language_set"], m["language_set"])
            and sets_compat(mentee["format_set"],   m["format_set"])]

def minmax(v, vmin, vmax):
    if vmax == vmin: return 0.5
    return round(max(0.0, min(1.0, (v - vmin) / (vmax - vmin))), 4)

def compute_score(mentee, mentor, weights):
    # skill: Jaccard. Пустой профиль → 0 (наказание, не исключение)
    sk_raw = jaccard(mentee["skills_list"], mentor["skills_list"])
    do_raw = domain_sim_raw(mentee, mentor)

    skill_val  = minmax(sk_raw, SKILL_MIN, SKILL_MAX) if sk_raw is not None else 0.0
    domain_val = minmax(do_raw, DOMAIN_MIN, DOMAIN_MAX)
    exp_val    = minmax(mentor["experience_norm"], EXP_MIN, EXP_MAX)
    rating_val = minmax((mentor["rating"] - 1) / 4, RATING_MIN, RATING_MAX)

    w_sk = float(weights["w_skills"])
    w_do = float(weights["w_domain"])
    w_ex = float(weights["w_exp"])
    w_ra = float(weights["w_rating"])

    score = w_sk*skill_val + w_do*domain_val + w_ex*exp_val + w_ra*rating_val

    breakdown = {
        "skill":  {"sim":skill_val,  "weight":w_sk,
                   "contribution":round(w_sk*skill_val,4),
                   "penalized": sk_raw is None},
        "domain": {"sim":domain_val, "weight":w_do,
                   "contribution":round(w_do*domain_val,4), "penalized":False},
        "exp":    {"sim":exp_val,    "weight":w_ex,
                   "contribution":round(w_ex*exp_val,4),    "penalized":False},
        "rating": {"sim":rating_val, "weight":w_ra,
                   "contribution":round(w_ra*rating_val,4), "penalized":False},
    }
    return round(score, 4), breakdown

BOOST_THRESHOLD = 0.30
TOP_K = 5

def apply_boost(score, mentor):
    if mentor["boosted"] and score >= BOOST_THRESHOLD:
        return round(score * (1 + mentor["boost_k"]), 4), True
    return score, False

print("Функции скоринга определены (4 фактора: skill, domain, exp, rating)")


Функции скоринга определены (4 фактора: skill, domain, exp, rating)


In [4]:
bounds = {r["factor"]:r for r in load_csv(os.path.join(BASE_DIR,"normalization_bounds.csv"))}
SKILL_MIN,  SKILL_MAX  = float(bounds["skill"]["min"]),  float(bounds["skill"]["max"])
DOMAIN_MIN, DOMAIN_MAX = float(bounds["domain"]["min"]), float(bounds["domain"]["max"])
EXP_MIN,    EXP_MAX    = float(bounds["exp"]["min"]),    float(bounds["exp"]["max"])
RATING_MIN, RATING_MAX = float(bounds["rating"]["min"]), float(bounds["rating"]["max"])
weights_by_id = {w["mentee_id"]:w
                 for w in load_csv(os.path.join(BASE_DIR,"mentee_weights.csv"))}
print("Границы и веса загружены")


Границы и веса загружены


In [5]:
BASE_W = {"w_skills":0.25,"w_domain":0.25,"w_exp":0.25,"w_rating":0.25}
SAMPLE_SIZE = 200
sample_mentees = random.sample(mentees, min(SAMPLE_SIZE,len(mentees)))

n_boosted = sum(1 for m in mentors if m["boosted"])
print(f"Менторов с бустом: {n_boosted} ({n_boosted/len(mentors)*100:.1f}%)")

boost_records = []
helped=0; not_helped=0; below_thresh=0
pool_sizes = []

for mentee in sample_mentees:
    cands = hard_filter(mentee, mentors)
    pool_sizes.append(len(cands))
    if not cands: continue
    w = weights_by_id.get(mentee["id"], BASE_W) if "weights_by_id" in dir() else BASE_W

    scored = [(m, compute_score(mentee,m,w)[0]) for m in cands]
    scored.sort(key=lambda x:-x[1])
    org_top5 = {m["id"] for m,_ in scored[:5]}

    bscored = sorted([(m,s,apply_boost(s,m)[0]) for m,s in scored], key=lambda x:-x[2])
    bst_top5 = {m["id"] for m,_,_ in bscored[:5]}

    for m,org,fin in bscored:
        if not m["boosted"]: continue
        if org < BOOST_THRESHOLD: below_thresh+=1; continue
        org_rank = next(i+1 for i,(mm,_) in enumerate(scored) if mm["id"]==m["id"])
        bst_rank = next(i+1 for i,(mm,_,__) in enumerate(bscored) if mm["id"]==m["id"])
        bhelped  = m["id"] in bst_top5 and m["id"] not in org_top5
        if bhelped: helped+=1
        else: not_helped+=1
        boost_records.append({
            "mentor_id":m["id"],"mentor_domain":m.get("domain",""),
            "mentor_level":m.get("level_raw",""),"mentor_rating":m["rating"],
            "boost_k":m["boost_k"],"organic_score":org,"final_score":fin,
            "score_delta":round(fin-org,4),"organic_rank":org_rank,
            "boosted_rank":bst_rank,"rank_improvement":org_rank-bst_rank,
            "pool_size":len(cands),"in_organic_top5":m["id"] in org_top5,
            "boost_helped":bhelped,
        })

total = helped+not_helped
avg_pool = sum(pool_sizes)/len(pool_sizes)
print(f"\nВыборка: {len(sample_mentees)} менти  |  Средний пул: {avg_pool:.0f}")
print(f"Бустированных в анализе: {total+below_thresh}")
print(f"  Ниже порога (не применён): {below_thresh}")
print(f"  Реально помог (→топ-5):    {helped}  ({helped/max(total,1)*100:.1f}%)")
print(f"  Уже были в органическом:   {not_helped}  ({not_helped/max(total,1)*100:.1f}%)")
applied = [r for r in boost_records]
if applied:
    ri = [r["rank_improvement"] for r in applied]
    print(f"  Среднее поднятие позиции:  {sum(ri)/len(ri):.0f} мест")
print("="*50)
if total > 0:
    if helped/total > 0.5: print("Буст эффективен — меняет выдачу чаще чем в половине случаев")
    elif helped/total > 0.1: print("Буст работает, но редко меняет выдачу — конкуренция высокая")
    else: print("Буст почти не влияет — сильные менторы и так в топ-5")
print("="*50)

with open(os.path.join(BASE_DIR,"boost_analytics.csv"),"w",newline="",encoding="utf-8") as f:
    fields=["mentor_id","mentor_domain","mentor_level","mentor_rating","boost_k",
            "organic_score","final_score","score_delta","organic_rank","boosted_rank",
            "rank_improvement","pool_size","in_organic_top5","boost_helped"]
    wcsv = csv.DictWriter(f,fieldnames=fields)
    wcsv.writeheader()
    wcsv.writerows(boost_records)
print(f"\nboost_analytics.csv  ({len(boost_records)} записей)")
print("Тетрадка 6 завершена!")


Менторов с бустом: 245 (12.2%)

Выборка: 200 менти  |  Средний пул: 856
Бустированных в анализе: 19994
  Ниже порога (не применён): 7590
  Реально помог (→топ-5):    292  (2.4%)
  Уже были в органическом:   12112  (97.6%)
  Среднее поднятие позиции:  80 мест
Буст почти не влияет — сильные менторы и так в топ-5

boost_analytics.csv  (12404 записей)
Тетрадка 6 завершена!
